In [17]:
import torch

## Simple self-attention mechanism

In [18]:
inputs  = torch.tensor(
    [[0.43, 0.15, 0.89], # Your        (x^1)
     [0.55, 0.87, 0.66], # journey     (x^2)
     [0.57, 0.85, 0.64], # starts      (x^3)
     [0.22, 0.58, 0.33], # with        (x^4)
     [0.77, 0.35, 0.10], # one         (x^5)
     [0.05, 0.80, 0.55]] # step        (x^6)
)

### Step 1) Compute attention scores $\omega_{ij}$.

In [19]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7940, 1.0865])


### Step 2a) Compute attention weights $\alpha_{ij}$.

In [20]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print(f"Attention weights {attn_weights_2_tmp}")
print(f"Attention weights sum {attn_weights_2_tmp.sum()}")

Attention weights tensor([0.1435, 0.2249, 0.2219, 0.1269, 0.1194, 0.1634])
Attention weights sum 1.0000001192092896


### Step 2b) Compute attention weights using softmax  function.

In [21]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print(f"Attention weights {attn_weights_2_naive}")
print(f"Attention weights sum {attn_weights_2_naive.sum()}")

Attention weights tensor([0.1372, 0.2356, 0.2310, 0.1228, 0.1169, 0.1566])
Attention weights sum 1.0


using softmax torch implementation

In [22]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(f"Attention weights {attn_weights_2}")
print(f"Attention weights sum {attn_weights_2.sum()}")

Attention weights tensor([0.1372, 0.2356, 0.2310, 0.1228, 0.1169, 0.1566])
Attention weights sum 1.0


### Step 3) Compute context vector $z^{(i)}$.

In [23]:
query = inputs[i]
context_vect_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vect_2 += attn_weights_2[i]*x_i
context_vect_2

tensor([0.4451, 0.6593, 0.5637])

Computing all context vectors:

1a. attention scores using `for` loops

In [10]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4726, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7940, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.8004, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.4054, 0.6565],
        [0.4726, 0.7940, 0.8004, 0.4054, 0.7254, 0.3735],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.3735, 0.9450]])

1b. using matrix multiplication

In [11]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4726, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7940, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.8004, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.4054, 0.6565],
        [0.4726, 0.7940, 0.8004, 0.4054, 0.7254, 0.3735],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.3735, 0.9450]])

2. attention weights

In [13]:
attn_weights = torch.softmax(attn_scores, dim=-1)
attn_weights

tensor([[0.2094, 0.2002, 0.1978, 0.1240, 0.1237, 0.1449],
        [0.1372, 0.2356, 0.2310, 0.1228, 0.1169, 0.1566],
        [0.1377, 0.2346, 0.2303, 0.1230, 0.1195, 0.1549],
        [0.1425, 0.2058, 0.2030, 0.1451, 0.1328, 0.1708],
        [0.1450, 0.2000, 0.2013, 0.1356, 0.1867, 0.1313],
        [0.1373, 0.2166, 0.2110, 0.1409, 0.1062, 0.1880]])

In [14]:
attn_weights.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [15]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4427, 0.6048, 0.5781],
        [0.4451, 0.6593, 0.5637],
        [0.4463, 0.6577, 0.5625],
        [0.4329, 0.6403, 0.5477],
        [0.4673, 0.6159, 0.5256],
        [0.4206, 0.6577, 0.5607]])

In [24]:
context_vect_2

tensor([0.4451, 0.6593, 0.5637])

## Self-attention mechanism with trainable weights

### Step 0) Define the ouput embedding size

In [25]:
x_2 = inputs[2]
d_in = 3
d_out = 2

### Step 1) Compute weight parameter matrices

In [26]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

### Step 2) Compute query, key and value vectors

In [28]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value 
query_2

tensor([0.4300, 1.4343])

In [29]:
keys = inputs @ W_key
values = inputs @ W_value
print(f"keys shape {keys.shape}")
print(f"Values shape {values.shape}")

keys shape torch.Size([6, 2])
Values shape torch.Size([6, 2])


### Step 3) Compute attention scores

In [30]:
keys_2 = keys[1]
attn_scores_22 = query_2.dot(keys_2)
attn_scores_22

tensor(1.8284)